In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os


In [ ]:
try:
    spark.stop()
except:
    pass


In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Procedure")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())


In [ ]:
silver_base_path = "../../data_lake/silver/silver_procedure_bundle/"
gold_base_path = "../../data_lake/gold/dim_procedure/"
gold_dimpatient = "../../data_lake/gold/dim_patient/"
gold_dimlocation = "../../data_lake/gold/dim_location/"
gold_dimdate = "../../data_lake/gold/dim_date/"


In [ ]:
df_procedure = spark.read.format("parquet").load(silver_base_path)
df_dimpatient = spark.read.format("parquet").load(gold_dimpatient)
df_dimlocation = spark.read.format("parquet").load(gold_dimlocation)
df_dimdate = spark.read.format("parquet").load(gold_dimdate)


In [ ]:
df_inter = (df_procedure.alias("proc")
    .join(df_dimpatient.alias("pat"), col("proc.patient_id") == col("pat.patient_id"), "left")
    .join(df_dimlocation.alias("loc"), col("proc.location_id") == col("loc.location_id"), "left")
    .join(df_dimdate.alias("d_start"), col("proc.performed_start_time").cast("date") == col("d_start.date"), "left")
    .join(df_dimdate.alias("d_end"), col("proc.performed_end_time").cast("date") == col("d_end.date"), "left")
    .select(
        F.conv(F.substring(F.md5(col("proc.procedure_id")), 1, 15), 16, 10).cast("bigint").alias("procedure_key"),
        col("proc.procedure_id"),
        col("pat.patient_key"),
        F.conv(F.substring(F.md5(col("proc.encounter_id")), 1, 15), 16, 10).cast("bigint").alias("encounter_key"),
        col("loc.location_key"),
        col("proc.status"),
        col("proc.code"),
        col("proc.code_display"),
        col("d_start.date_key").alias("performed_start_date_key"),
        col("d_end.date_key").alias("performed_end_date_key"),
        col("proc.performed_start_time"),
        col("proc.performed_end_time"),
        col("proc.reason_code"),
        col("proc.reason_code_display"),
        col("proc.reason_reference"),
        col("proc.reason_reference_display"),
        F.current_timestamp().alias("gold_timestamp")
    )
)


In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)


In [ ]:
spark.stop()
